# P0-3 · pandas: làm sạch và tổng hợp bảng dữ liệu

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import pandas as pd
s = pd.Series([1500, 400, 450], index=["Bàn A", "Ghế B", "Ghế C"], name="price")
s["Ghế B"]           # lấy theo nhãn: 400
s[s > 420]           # lọc bằng mặt nạ như numpy: Bàn A và Ghế C

400
Bàn A    1500
Ghế C     450


In [ ]:
import io
import pandas as pd

csv_text = """date,product,qty,price
2026-01-03,Bàn A,2,1500
2026-01-15, ghế b,1,400
2026-02-02,Bàn A,1,
2026-02-20,Bàn A,3,"1,500"
2026-02-20,Bàn A,3,"1,500"
2026-02-21,Ghế B,2,450
"""
raw = pd.read_csv(io.StringIO(csv_text))    # file thật: pd.read_csv("orders.csv"), pd.read_excel("orders.xlsx")
raw

         date product  qty  price
0  2026-01-03   Bàn A    2   1500
1  2026-01-15   ghế b    1    400
2  2026-02-02   Bàn A    1    NaN
3  2026-02-20   Bàn A    3  1,500
4  2026-02-20   Bàn A    3  1,500
5  2026-02-21   Ghế B    2    450


In [ ]:
raw.shape                        # (6, 4): 6 dòng, 4 cột
raw.dtypes                       # price là kiểu chữ vì có "1,500"; date cũng đang là chữ
raw.isna().sum()                 # số ô trống từng cột: price có 1
raw.duplicated().sum()           # 1 dòng trùng hoàn toàn với một dòng phía trên
raw["product"].value_counts()    # " ghế b" và "Ghế B" bị đếm thành hai sản phẩm

(6, 4)
date         str
product      str
qty        int64
price        str
date       0
product    0
qty        0
price      1
1
product
Bàn A     4
 ghế b    1
Ghế B     1


In [ ]:
df = raw.drop_duplicates().reset_index(drop=True)              # bỏ dòng trùng, đánh lại index 0, 1, 2...
df["product"] = df["product"].str.strip().str.title()          # " ghế b" → "Ghế B"
df["price"] = pd.to_numeric(df["price"].str.replace(",", ""))  # "1,500" → 1500.0; ô trống vẫn là NaN
df["date"] = pd.to_datetime(df["date"])                        # chữ → kiểu ngày, dùng được .dt.month
df

        date product  qty   price
0 2026-01-03   Bàn A    2  1500.0
1 2026-01-15   Ghế B    1   400.0
2 2026-02-02   Bàn A    1     NaN
3 2026-02-20   Bàn A    3  1500.0
4 2026-02-21   Ghế B    2   450.0


In [ ]:
df.dtypes                         # date là datetime64, price là float64
df["product"].value_counts()      # còn đúng hai sản phẩm: Bàn A 3, Ghế B 2

date       datetime64[us]
product               str
qty                 int64
price             float64
product
Bàn A    3
Ghế B    2


In [ ]:
df.sort_values("qty", ascending=False)     # sắp theo qty giảm dần
df.nlargest(2, "qty")                      # 2 dòng có qty lớn nhất
df["product"].nunique()                    # 2: số giá trị khác nhau
df["product"].value_counts(normalize=True) # tỉ lệ thay vì số đếm: Bàn A 0.6, Ghế B 0.4

        date product  qty   price
3 2026-02-20   Bàn A    3  1500.0
0 2026-01-03   Bàn A    2  1500.0
4 2026-02-21   Ghế B    2   450.0
2 2026-02-02   Bàn A    1     NaN
1 2026-01-15   Ghế B    1   400.0
        date product  qty   price
3 2026-02-20   Bàn A    3  1500.0
0 2026-01-03   Bàn A    2  1500.0
2
product
Bàn A    0.6
Ghế B    0.4


In [ ]:
df[["product", "qty"]]                     # nhiều cột → DataFrame
df.loc[df["qty"] >= 2, ["product", "qty"]] # loc: lọc theo điều kiện và tên cột
df.iloc[0:2, 1:3]                          # iloc: theo vị trí, giống slicing của numpy

  product  qty
0   Bàn A    2
1   Ghế B    1
2   Bàn A    1
3   Bàn A    3
4   Ghế B    2
  product  qty
0   Bàn A    2
3   Bàn A    3
4   Ghế B    2
  product  qty
0   Bàn A    2
1   Ghế B    1


In [ ]:
len(df[df["qty"] >= 2 & (df["price"] > 1000)])      # sai: ra 5 dòng
len(df[(df["qty"] >= 2) & (df["price"] > 1000)])    # đúng: ra 2 dòng

5
2


In [ ]:
df["price"] = df["price"].fillna(df.groupby("product")["price"].transform("median"))
df["revenue"] = df["qty"] * df["price"]     # cột mới, tính trên cả cột (vector hóa)
df

        date product  qty   price  revenue
0 2026-01-03   Bàn A    2  1500.0   3000.0
1 2026-01-15   Ghế B    1   400.0    400.0
2 2026-02-02   Bàn A    1  1500.0   1500.0
3 2026-02-20   Bàn A    3  1500.0   4500.0
4 2026-02-21   Ghế B    2   450.0    900.0


In [ ]:
df.groupby("product")["revenue"].sum()
# Bàn A    9000.0      ← 3000 + 1500 + 4500
# Ghế B    1300.0      ← 400 + 900

product
Bàn A    9000.0
Ghế B    1300.0


In [ ]:
df.groupby("product").agg(
    so_don=("qty", "size"),       # số dòng mỗi nhóm
    tong_sl=("qty", "sum"),
    doanh_thu=("revenue", "sum"),
)

         so_don  tong_sl  doanh_thu
product                            
Bàn A         3        6     9000.0
Ghế B         2        3     1300.0


In [ ]:
df.groupby(df["date"].dt.to_period("M"))["revenue"].sum()
# 2026-01    3400.0
# 2026-02    6900.0

date
2026-01    3400.0
2026-02    6900.0
Freq: M


In [ ]:
df.pivot_table(index=df["date"].dt.month, columns="product",
               values="revenue", aggfunc="sum", fill_value=0)

product   Bàn A  Ghế B
date                  
1        3000.0  400.0
2        6000.0  900.0


In [ ]:
products = pd.DataFrame({"product": ["Bàn A", "Ghế B"], "category": ["Bàn", "Ghế"]})
full = df.merge(products, on="product", how="left")   # thêm cột category vào mỗi đơn
full[["date", "product", "category"]]

        date product category
0 2026-01-03   Bàn A      Bàn
1 2026-01-15   Ghế B      Ghế
2 2026-02-02   Bàn A      Bàn
3 2026-02-20   Bàn A      Bàn
4 2026-02-21   Ghế B      Ghế


In [ ]:
jan = df[df["date"].dt.month == 1]
feb = df[df["date"].dt.month == 2]
pd.concat([jan, feb], ignore_index=True).shape    # (5, 5): lại đủ 5 đơn

(5, 5)


In [ ]:
dup = pd.DataFrame({"product": ["Bàn A", "Bàn A"], "color": ["trắng", "đen"]})
len(df), len(df.merge(dup, on="product", how="left"))   # (5, 8)

(5, 8)


In [ ]:
df["month"] = df["date"].dt.month                     # [1, 1, 2, 2, 2]
df["weekday"] = df["date"].dt.dayofweek               # 0 = thứ Hai, ..., 6 = Chủ nhật
df["is_table"] = df["product"].str.contains("Bàn")    # True/False
df["size"] = pd.cut(df["revenue"], bins=[0, 1000, 3000, 10_000], labels=["nhỏ", "vừa", "lớn"])
df[["product", "revenue", "month", "weekday", "is_table", "size"]]

  product  revenue  month  weekday  is_table size
0   Bàn A   3000.0      1        5      True  vừa
1   Ghế B    400.0      1        3     False  nhỏ
2   Bàn A   1500.0      2        0      True  vừa
3   Bàn A   4500.0      2        4      True  lớn
4   Ghế B    900.0      2        5     False  nhỏ


In [ ]:
import numpy as np, time
big = pd.DataFrame({"qty": np.random.default_rng(0).integers(1, 5, 200_000), "price": 450.0})
t = time.time(); r1 = big.apply(lambda row: row["qty"] * row["price"], axis=1); t1 = time.time() - t
t = time.time(); r2 = big["qty"] * big["price"]; t2 = time.time() - t
print(f"apply: {t1:.2f} s, vector hóa: {t2 * 1000:.1f} ms, cùng kết quả: {r1.equals(r2)}")

apply: 0.82 s, vector hóa: 1.0 ms, cùng kết quả: True


In [ ]:
df.to_csv("orders_clean.csv", index=False)       # index=False: không ghi cột số thứ tự
pd.read_csv("orders_clean.csv", parse_dates=["date"]).shape   # đọc lại đủ (5, 9)

(5, 9)


## Bài tập 3: Làm sạch tên sản phẩm {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import pandas as pd
names = pd.Series([" Bàn A", "bàn a", "BÀN A ", "Ghế B", "ghế  b", None])

In [ ]:
import pandas as pd
names = pd.Series([" Bàn A", "bàn a", "BÀN A ", "Ghế B", "ghế  b", None])
clean = names.str.strip().str.split().str.join(" ").str.title()
print(clean.value_counts())
print("thiếu tên:", clean.isna().sum())

Bàn A    3
Ghế B    2
Name: count, dtype: int64
thiếu tên: 1


## Bài tập 4: Doanh thu theo tháng và tỉ lệ hoàn {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import pandas as pd
orders = pd.DataFrame({
    "date": pd.to_datetime(["2026-03-02", "2026-03-15", "2026-03-20", "2026-04-01", "2026-04-09", "2026-04-28"]),
    "product": ["Bàn A", "Gương", "Gương", "Bàn A", "Gương", "Bàn A"],
    "qty": [1, 2, 1, 2, 1, 1],
    "price": [1500, 300, 300, 1500, 300, 1500],
    "returned": [False, True, False, False, False, True],
})

In [ ]:
orders["revenue"] = orders["qty"] * orders["price"]
sold = orders[~orders["returned"]]
print(sold.groupby(sold["date"].dt.to_period("M"))["revenue"].sum())
print(orders.groupby("product")["returned"].mean().sort_values(ascending=False))

date
2026-03    1800
2026-04    3300
Freq: M, Name: revenue, dtype: int64
product
Bàn A    0.333333
Gương    0.333333
Name: returned, dtype: float64
